# Génération des scénarios EnergyScope

Pipeline en 4 étapes :

1. **Demande de base 2023** — calculée depuis `demands_inputs.xlsx` (matrices HHV par secteur).
2. **Facteurs de croissance** — calculés de façon générique depuis l'onglet `mapping` de
   `scenario_inputs.xlsx` (plus de dictionnaire de correspondance codé en dur dans le notebook).
3. **Inputs de scénario additionnels** (CO2, capture, séquestration, ...) — lus via un
   **registre** (`SCENARIO_EXTRA_PARAMS`). Pour ajouter un nouvel input plus tard :
   une ligne dans ce registre (+ un onglet Excel si besoin), rien d'autre à changer.
4. **Écriture du `.dat`** — demande + tous les paramètres du registre.

Pour ajouter un nouveau scénario : dupliquer `scenario_inputs.xlsx` (ou ajouter une ligne à
l'onglet `scenario`), pas besoin de toucher ce notebook.

In [1]:
import pandas as pd
from pathlib import Path

DEMANDS_PATH = Path("demands_inputs.xlsx")
SCENARIO_PATH = Path("scenario_inputs.xlsx")
OUTPUT_DIR = Path("../data_constraints")

PJ_TO_GWH = 1_000_000 / 3600  # 1 PJ = 1e6/3600 GWh
pd.options.display.float_format = lambda x: f"{x:,.2f}"

## 1. Demande de base 2023 (`demands_inputs.xlsx`)

Une fonction par secteur, identique à la logique d'origine du notebook — juste nettoyée en
fonctions réutilisables (plus de cellules dupliquées).

In [2]:
def industry_demand_from_hhv(ind_hhv, path=DEMANDS_PATH):
    ind_hhv = ind_hhv.copy()
    ind_hhv.index.name, ind_hhv.columns.name = "fuel", "industry"

    ind_fuel = pd.read_excel(path, "industry_fuel_params", index_col=0).reindex(ind_hhv.index)
    ind_par = pd.read_excel(path, "industry_params", index_col=0).reindex(ind_hhv.columns)
    ind_const = pd.read_excel(path, "industry_constants", index_col=0)["value"].to_dict()

    ind_final = ind_hhv.mul(ind_fuel["lhv_hhv_ratio"], axis=0).mul(ind_fuel["efficiency"], axis=0)
    elec_by_ind = ind_final.loc["Electricity"]
    heat_by_ind = ind_final.sum(axis=0) - elec_by_ind
    elec_total = elec_by_ind.sum()

    volt = ind_par["voltage_class"]
    share = {lvl: elec_by_ind[volt == lvl].sum() / elec_total for lvl in ("MV", "HV", "EHV")}
    fp, fl = ind_const["electricity_power_fraction"], ind_const["lighting_fraction"]

    eud = {
        "ELECTRICITY_EHV": elec_total * fp * share["EHV"],
        "ELECTRICITY_HV": elec_total * fp * share["HV"],
        "ELECTRICITY_MV": elec_total * fp * share["MV"],
        "LIGHTING": elec_total * fl,
        "HEAT_HIGH_T": (heat_by_ind * ind_par["ht_pct"] / 100).sum(),
        "HEAT_LOW_T_SH": (heat_by_ind * ind_par["sh_pct"] / 100).sum(),
        "HEAT_LOW_T_HW": (heat_by_ind * ind_par["hw_pct"] / 100).sum(),
    }
    return pd.DataFrame(
        [(k, "INDUSTRY", v * PJ_TO_GWH, "GWh") for k, v in eud.items()],
        columns=["eud_cat", "sector", "value_ES", "unit_ES"],
    )

In [3]:
def agriculture_demand(path=DEMANDS_PATH):
    agr = pd.read_excel(path, "agriculture_fuels", index_col=0)
    a = pd.read_excel(path, "agriculture_constants", index_col=0)["value"].to_dict()
    agr["final_pj"] = agr["hhv_pj"] * agr["lhv_hhv_ratio"] * agr["efficiency"]

    mobility_hhv = agr.loc[agr["role"] == "mobility", "hhv_pj"].sum()
    electricity_pj = agr.loc[agr["role"] == "electricity", "final_pj"].sum()
    heat_pj = agr.loc[agr["role"] == "heat", "final_pj"].sum()

    return pd.DataFrame([
        ("MOBILITY_FREIGHT_SD", "AGRICULTURE", mobility_hhv * a["freight_conv_mtkm_per_pj"], "Mtkm"),
        ("ELECTRICITY_MV", "AGRICULTURE", electricity_pj * a["elec_mv_fraction"] * PJ_TO_GWH, "GWh"),
        ("LIGHTING", "AGRICULTURE", electricity_pj * a["elec_lighting_fraction"] * PJ_TO_GWH, "GWh"),
        ("HEAT_LOW_T_SH", "AGRICULTURE", heat_pj * a["heat_sh_fraction"] * PJ_TO_GWH, "GWh"),
        ("HEAT_LOW_T_HW", "AGRICULTURE", heat_pj * a["heat_hw_fraction"] * PJ_TO_GWH, "GWh"),
    ], columns=["eud_cat", "sector", "value_ES", "unit_ES"])

In [4]:
def residential_demand(path=DEMANDS_PATH):
    res_hhv = pd.read_excel(path, "residential_hhv_matrix", index_col=0)
    res_hhv.index.name, res_hhv.columns.name = "equipment", "service"
    res_par = pd.read_excel(path, "residential_equip_params", index_col=0).reindex(res_hhv.index)

    res_useful = res_hhv.mul(res_par["lhv_hhv_ratio"], axis=0).mul(res_par["efficiency_pct"] / 100, axis=0)
    res_by_service = res_useful.sum(axis=0)

    res_map = {
        "space_heating": "HEAT_LOW_T_SH",
        "water_heating": "HEAT_LOW_T_HW",
        "appliances": "ELECTRICITY_LV",
        "space_cooling": "HEAT_LOW_T_SC",
        "lighting": "LIGHTING",
    }
    return pd.DataFrame(
        [(eud, "HOUSEHOLDS", res_by_service[svc] * PJ_TO_GWH, "GWh") for svc, eud in res_map.items()],
        columns=["eud_cat", "sector", "value_ES", "unit_ES"],
    )

In [5]:
def services_demand(srv_elec, srv_hhv, path=DEMANDS_PATH):
    by_volt = srv_elec.groupby("voltage_class")["electricity_pj"].sum()

    srv_hhv = srv_hhv.copy()
    srv_hhv.index.name, srv_hhv.columns.name = "fuel", "service"
    srv_par = pd.read_excel(path, "services_fuel_params", index_col=0).reindex(srv_hhv.index)

    srv_final = srv_hhv.mul(srv_par["lhv_hhv_ratio"], axis=0).mul(srv_par["efficiency"], axis=0)
    srv_by_service = srv_final.sum(axis=0)

    aux = ["aux_equipments", "aux_motors"]
    aux_non_elec = srv_by_service[aux].sum() - srv_final.loc["Electricity", aux].sum()

    return pd.DataFrame([
        ("ELECTRICITY_LV", "SERVICES", by_volt["LV"] * PJ_TO_GWH, "GWh"),
        ("ELECTRICITY_MV", "SERVICES", by_volt["MV"] * PJ_TO_GWH, "GWh"),
        ("LIGHTING", "SERVICES", (srv_by_service["lighting"] + srv_by_service["street_lighting"]) * PJ_TO_GWH, "GWh"),
        ("HEAT_LOW_T_SC", "SERVICES", srv_by_service["space_cooling"] * PJ_TO_GWH, "GWh"),
        ("HEAT_HIGH_T", "SERVICES", aux_non_elec * PJ_TO_GWH, "GWh"),
        ("HEAT_LOW_T_SH", "SERVICES", srv_by_service["space_heating"] * PJ_TO_GWH, "GWh"),
        ("HEAT_LOW_T_HW", "SERVICES", srv_by_service["water_heating"] * PJ_TO_GWH, "GWh"),
    ], columns=["eud_cat", "sector", "value_ES", "unit_ES"])

In [6]:
def transport_demand(veh, path=DEMANDS_PATH):
    psplit = pd.read_excel(path, "transport_passenger_split", index_col=0)
    fsplit = pd.read_excel(path, "transport_freight_split", index_col=0)

    assert (psplit.sum(axis=1).round(6) == 1).all(), "parts passagers != 1"
    assert (fsplit.sum(axis=1).round(6) == 1).all(), "parts fret != 1"

    pass_act = veh.loc[psplit.index, "activity"]
    frt_act = veh.loc[fsplit.index, "activity"]

    p = psplit.mul(pass_act, axis=0).sum(axis=0)
    f = fsplit.mul(frt_act, axis=0).sum(axis=0)

    return pd.DataFrame([
        ("MOBILITY_PASSENGER_SD", "TRANSPORTATION", p["SD"], "Mpkm"),
        ("MOBILITY_PASSENGER_MD", "TRANSPORTATION", p["MD"], "Mpkm"),
        ("MOBILITY_PASSENGER_LD", "TRANSPORTATION", p["LD"], "Mpkm"),
        ("MOBILITY_FREIGHT_SD", "TRANSPORTATION", f["SD"], "Mtkm"),
        ("MOBILITY_FREIGHT_MD", "TRANSPORTATION", f["MD"], "Mtkm"),
        ("MOBILITY_FREIGHT_LD", "TRANSPORTATION", f["LD"], "Mtkm"),
        ("MOBILITY_FREIGHT_ELD", "TRANSPORTATION", f["ELD"], "Mtkm"),
    ], columns=["eud_cat", "sector", "value_ES", "unit_ES"])

In [7]:
def base_2023_tables(path=DEMANDS_PATH):
    """Toutes les tables sectorielles 2023, calculées depuis demands_inputs.xlsx."""
    ind_hhv = pd.read_excel(path, "industry_hhv_matrix", index_col=0)
    veh = pd.read_excel(path, "transport_vehicles", index_col=0)
    srv_elec = pd.read_excel(path, "services_electricity", index_col=0)
    srv_hhv = pd.read_excel(path, "services_hhv_matrix", index_col=0)

    tables = {
        "INDUSTRY": industry_demand_from_hhv(ind_hhv, path),
        "AGRICULTURE": agriculture_demand(path),
        "HOUSEHOLDS": residential_demand(path),
        "SERVICES": services_demand(srv_elec, srv_hhv, path),
        "TRANSPORTATION": transport_demand(veh, path),
    }
    raw = {"industry": ind_hhv, "transport": veh,
           "services_electricity": srv_elec, "services_hhv": srv_hhv}
    return tables, raw

In [8]:
base_tables, _raw = base_2023_tables()
pd.concat(base_tables.values(), ignore_index=True)

,eud_cat,sector,value_ES,unit_ES
0,ELECTRICITY_EHV,INDUSTRY,"6,145.77",GWh
1,ELECTRICITY_HV,INDUSTRY,"69,317.43",GWh
2,ELECTRICITY_MV,INDUSTRY,"11,405.86",GWh
3,LIGHTING,INDUSTRY,"5,544.83",GWh
4,HEAT_HIGH_T,INDUSTRY,"72,060.04",GWh
5,HEAT_LOW_T_SH,INDUSTRY,"3,258.18",GWh
6,HEAT_LOW_T_HW,INDUSTRY,620.00,GWh
7,MOBILITY_FREIGHT_SD,AGRICULTURE,"3,147.02",Mtkm
8,ELECTRICITY_MV,AGRICULTURE,"1,976.53",GWh
9,LIGHTING,AGRICULTURE,82.36,GWh


## 2. Facteurs de croissance base → scénario cible (`scenario_inputs.xlsx`)

Depuis la réorganisation en blocs, un scénario n'est plus une colonne de valeurs
directes : c'est une **combinaison de variantes**, assemblée dans l'onglet `scenario`
(une colonne par bloc : `transport`, `industry`, `agriculture`, `residential`,
`services`, `capacite`, `contrainte_env` — chacune pointant vers le nom d'une
variante définie dans l'onglet correspondant). `get_scenario_assembly` lit cette
ligne d'assemblage ; toutes les fonctions ci-dessous s'en servent pour aller
chercher, secteur par secteur, la bonne variante plutôt que le nom du scénario
directement.

In [9]:
SECTOR_KEY = {
    "transport": "TRANSPORTATION",
    "industry": "INDUSTRY",
    "agriculture": "AGRICULTURE",
    "residential": "HOUSEHOLDS",
    "services": "SERVICES",
}

BASE_SCENARIO = "2023"  # variante de référence (dans chaque onglet sectoriel) pour tous les ratios de croissance


def _read_scenario_sheet(scenario_path, sheet):
    """Lit un onglet indexé par nom de VARIANTE (colonne 'variante' pour les onglets
    sectoriels/capacites/contrainte_env, ou 'scenario' pour la table d'assemblage), et
    normalise l'index en texte : les onglets mélangent des noms numériques (2023) et
    textuels ('Haute demande'), ce qui donne un index pandas de type 'object' -- on
    force tout en str pour que les .loc[...] soient fiables quel que soit le nom."""
    df = pd.read_excel(scenario_path, sheet_name=sheet, index_col=0)
    df.index = df.index.map(lambda x: str(int(x)) if isinstance(x, float) and x.is_integer() else str(x))
    return df


def get_scenario_assembly(scenario_name, scenario_path=SCENARIO_PATH):
    """Lit l'onglet `scenario` (table d'assemblage) et retourne la ligne du scénario
    demandé. Chaque colonne (transport, industry, agriculture, residential, services,
    capacite, contrainte_env) contient le nom de la VARIANTE choisie pour ce bloc."""
    df = _read_scenario_sheet(scenario_path, "scenario")
    if scenario_name not in df.index:
        raise KeyError(f"Scénario '{scenario_name}' introuvable dans l'onglet 'scenario'.")
    return df.loc[scenario_name]


def growth_ratios_by_demand_entity(scenario_name, scenario_path=SCENARIO_PATH):
    """Pour chaque (secteur, demands_en) du mapping, calcule le ratio
    variante_choisie / BASE_SCENARIO, où la variante choisie pour chaque secteur
    vient de l'onglet `scenario` (table d'assemblage). Retourne {sector_ES: {demands_en: ratio}}."""
    mapping = pd.read_excel(scenario_path, sheet_name="mapping")
    units = pd.read_excel(scenario_path, sheet_name="units_demand")
    unit_of = units.set_index(["sector", "entity"])["unit"]

    drivers = {s: _read_scenario_sheet(scenario_path, s) for s in SECTOR_KEY}
    assembly_row = get_scenario_assembly(scenario_name, scenario_path)

    out = {sector_es: {} for sector_es in SECTOR_KEY.values()}
    for (sector, target), grp in mapping.groupby(["sector", "demands_en"]):
        cols = grp["driver_fr"].tolist()
        us = {unit_of.get((sector, c)) for c in cols}
        if len(us) > 1:
            print(f"[avertissement] unités mixtes pour {sector}/{target} : {us} "
                  f"— ratio calculé par somme brute (approximation)")

        df = drivers[sector]
        variante = assembly_row[sector]  # nom de la variante choisie pour ce secteur
        total = df[cols].sum(axis=1)
        v0, v1 = total.loc[BASE_SCENARIO], total.loc[variante]
        out[SECTOR_KEY[sector]][target] = v1 / v0 if v0 else float("nan")

    return out


### Application des ratios par secteur

- **Transport** : les noms de véhicules dans `demands_inputs.xlsx` sont déjà dans la
  nomenclature `demands_en` → on applique le ratio directement par véhicule, puis on
  recalcule la répartition par classe de distance.
- **Industry** : ratio appliqué par industrie mappée (colonnes de `industry_hhv_matrix`).
- **Services / Residential** : un seul ratio global (le detail HHV n'a pas de correspondance
  fine côté drivers).
- **Agriculture** : pas de correspondance driver ↔ demande (axes incompatibles, cf. README
  de `scenario_inputs.xlsx`) → non mis à l'échelle, comme dans le notebook d'origine.

In [10]:
def scaled_scenario_tables(scenario_name, demands_path=DEMANDS_PATH, scenario_path=SCENARIO_PATH):
    """Construit les tables de demande ES pour un scénario donné (ex. 'MEIE_2050', 'MEIE_2050_HD').
    Le scénario est une combinaison de VARIANTES (une par secteur), lues dans l'onglet
    `scenario` (table d'assemblage) ; l'année AMPL cible (pour le tag 'YEAR_...' dans
    le .dat) vient de la colonne `year` de ce même onglet."""
    base_tables, raw = base_2023_tables(demands_path)
    growth = growth_ratios_by_demand_entity(scenario_name, scenario_path)
    assembly_row = get_scenario_assembly(scenario_name, scenario_path)

    # --- Transport ---
    entity_ratio_t = growth["TRANSPORTATION"]
    veh = raw["transport"].copy()
    veh["ratio"] = [entity_ratio_t.get(v, 1.0) for v in veh.index]
    veh["activity"] = veh["activity"] * veh["ratio"]
    transport_scaled = transport_demand(veh, demands_path)

    # --- Industry ---
    ind_hhv = raw["industry"].copy()
    entity_ratio_ind = growth["INDUSTRY"]
    for col in ind_hhv.columns:
        ind_hhv[col] = ind_hhv[col] * entity_ratio_ind.get(col, 1.0)
    industry_scaled = industry_demand_from_hhv(ind_hhv, demands_path)

    # --- Services (ratio global) ---
    services_ratio = _read_scenario_sheet(scenario_path, "services")
    services_variante = assembly_row["services"]
    services_ratio_global = (services_ratio.sum(axis=1).loc[services_variante]
                              / services_ratio.sum(axis=1).loc[BASE_SCENARIO])
    srv_elec_scaled = raw["services_electricity"].copy()
    srv_elec_scaled["electricity_pj"] = srv_elec_scaled["electricity_pj"] * services_ratio_global
    srv_hhv_scaled = raw["services_hhv"] * services_ratio_global
    services_scaled = services_demand(srv_elec_scaled, srv_hhv_scaled, demands_path)

    # --- Residential (ratio global) ---
    res_ratio = _read_scenario_sheet(scenario_path, "residential")
    res_variante = assembly_row["residential"]
    res_ratio_global = (res_ratio.sum(axis=1).loc[res_variante]
                         / res_ratio.sum(axis=1).loc[BASE_SCENARIO])
    residential_scaled = base_tables["HOUSEHOLDS"].copy()
    residential_scaled["value_ES"] = residential_scaled["value_ES"] * res_ratio_global

    # --- Agriculture (non mise à l'échelle) ---
    agriculture_scaled = base_tables["AGRICULTURE"]

    return {
        "TRANSPORTATION": transport_scaled,
        "INDUSTRY": industry_scaled,
        "SERVICES": services_scaled,
        "HOUSEHOLDS": residential_scaled,
        "AGRICULTURE": agriculture_scaled,
    }


def end_uses_demand_matrix(tables):
    end_uses = pd.concat(list(tables.values()), ignore_index=True)
    end_uses = end_uses[["sector", "eud_cat", "value_ES", "unit_ES"]]
    return end_uses.pivot_table(index="eud_cat", columns="sector", values="value_ES", aggfunc="sum")


## 3. Inputs de scénario additionnels : contrainte environnementale (`contrainte_env`)

Comme pour la demande, la contrainte CO2/capture/séquestration à appliquer n'est
plus lue directement par nom de scénario : `load_scenario_extra_params` va chercher
la variante choisie dans la colonne `contrainte_env` de l'onglet `scenario`, puis va
lire cette variante dans l'onglet `contrainte_env`.

In [11]:
SCENARIO_EXTRA_PARAMS = {
    # nom du paramètre AMPL : (onglet Excel, colonne)
    "co2_limit": ("contrainte_env", "CO2 limit [Mt]"),
    "ccs_capture_limit": ("contrainte_env", "Capture limite [Mt]"),
    "ccs_sequestration_limit": ("contrainte_env", "Sequestration limite [Mt]"),
}


def load_scenario_extra_params(scenario_name, scenario_path=SCENARIO_PATH):
    """Lit tous les paramètres du registre pour la variante `contrainte_env` choisie
    pour ce scénario (via l'onglet `scenario`). Une variante manquante ou une valeur
    manquante est ignorée avec un avertissement (pas de blocage)."""
    assembly_row = get_scenario_assembly(scenario_name, scenario_path)
    variante = assembly_row.get("contrainte_env")

    values = {}
    if pd.isna(variante):
        print(f"[avertissement] contrainte_env : aucune variante choisie pour '{scenario_name}' — ignoré")
        return values

    cache = {}
    for param_name, (sheet, column) in SCENARIO_EXTRA_PARAMS.items():
        if sheet not in cache:
            cache[sheet] = _read_scenario_sheet(scenario_path, sheet)
        df = cache[sheet]
        if variante not in df.index or column not in df.columns:
            print(f"[avertissement] {param_name} : pas de valeur pour la variante '{variante}' "
                  f"dans '{sheet}'/'{column}' — ignoré")
            continue
        values[param_name] = df.loc[variante, column]
    return values


In [12]:
extra_2050_hd = load_scenario_extra_params("MEIE_2050_HD")
extra_2050_hd


{'co2_limit': np.int64(0),
 'ccs_capture_limit': np.int64(25),
 'ccs_sequestration_limit': np.int64(25)}

## 4. Écriture du fichier `.dat`

In [13]:
COMMENTED_PARAMS = {}


def generer_dat_scenario(nom_fichier, annee, demand_df, extra_params=None):
    """
    nom_fichier   : chemin de sortie
    annee         : année cible (ex. 2050) -> tag 'YEAR_2050' (utilisé pour la demande)
    demand_df     : matrice de demande (eud_cat en index, secteurs en colonnes)
    extra_params  : dict {nom_param_AMPL: valeur} — paramètres scalaires (non indexés
                    par année). Ceux listés dans COMMENTED_PARAMS sont écrits en
                    commentaire (`# let ...`) plutôt qu'activés.
    """
    df_temp = demand_df.copy()
    if "eud_cat" in df_temp.columns:
        df_temp = df_temp.set_index("eud_cat")

    secteurs = sorted(df_temp.columns)
    year_tag = f"YEAR_{annee}"

    with open(nom_fichier, "w", encoding="utf-8") as f:
        if extra_params:
            f.write("# CO2 emissions and capture\n")
            for param_name, valeur in extra_params.items():
                prefix = "# " if param_name in COMMENTED_PARAMS else ""
                f.write(f"{prefix}let {param_name} := {valeur:.2f} ;\n")
            f.write("\n")

        for sector in secteurs:
            serie = df_temp[sector].dropna()
            if serie.empty:
                continue
            f.write(f"# {sector.upper()}\n")
            for end_use, value in serie.items():
                f.write(f"let end_uses_demand_year['{year_tag}','{end_use}','{sector}'] := {value:.2f} ;\n")

    print(f"'{nom_fichier}' généré pour l'année {annee}.")


def generate_scenario_dat(scenario_name, year, output_dir=OUTPUT_DIR,
                            demands_path=DEMANDS_PATH, scenario_path=SCENARIO_PATH):
    """scenario_name : clé dans les onglets (ex. '2050_HD') ; year : année AMPL cible
    (ex. 2050, venant de la colonne 'year' de l'onglet scenario) -> tag 'YEAR_2050'."""
    tables = scaled_scenario_tables(scenario_name, demands_path, scenario_path)
    matrix = end_uses_demand_matrix(tables)
    extra = load_scenario_extra_params(scenario_name, scenario_path)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{scenario_name}.dat"   # <-- avant : f"{scenario_name}_{year}.dat"
    generer_dat_scenario(out_path, year, matrix, extra)
    return out_path

### Génération pour tous les scénarios de l'onglet `scenario`

Le nom du scénario (index de l'onglet `scenario`) sert de clé pour aller chercher,
via `get_scenario_assembly`, la variante choisie pour chaque bloc (secteurs de
demande, `contrainte_env`). La colonne `year` donne l'année AMPL à utiliser dans le
`.dat` (`MEIE_2050` et `MEIE_2050_HD` produisent tous deux `YEAR_2050`, avec des
valeurs de demande différentes selon les variantes assemblées).

In [14]:
scenarios = pd.read_excel(SCENARIO_PATH, sheet_name="scenario", index_col=0)
scenarios.index = scenarios.index.map(str)
scenarios

,year,transport,industry,agriculture,residential,services,capacite,contrainte_env,description
scenario,,,,,,,,,
2023,2023,2023,2023,2023,2023,2023,Statu quo,NaN,Scenario de base qui a servi à calibrer le mod...
MEIE_2050,2050,Croissance modérée,Croissance modérée,Croissance modérée,Croissance modérée,Croissance modérée,Statu quo,carboneutralité,Scenario principale du MEIE
MEIE_2050_HD,2050,Haute demande,Haute demande,Haute demande,Haute demande,Haute demande,Statu quo,carbo_CCS,Forte demande (High Demand)
MEIE_2050_no_CCS,2050,Croissance modérée,Croissance modérée,Croissance modérée,Croissance modérée,Croissance modérée,Développement Hydro,carbo_no_CCS,Nan


In [15]:
for scenario_name, row in scenarios.iterrows():
    generate_scenario_dat(scenario_name=scenario_name, year=int(row["year"]))

[avertissement] unités mixtes pour industry/Chemicals : {'kt', 'GO (M$ 2016)'} — ratio calculé par somme brute (approximation)
[avertissement] contrainte_env : aucune variante choisie pour '2023' — ignoré
'..\data_constraints\2023.dat' généré pour l'année 2023.
[avertissement] unités mixtes pour industry/Chemicals : {'kt', 'GO (M$ 2016)'} — ratio calculé par somme brute (approximation)
'..\data_constraints\MEIE_2050.dat' généré pour l'année 2050.
[avertissement] unités mixtes pour industry/Chemicals : {'kt', 'GO (M$ 2016)'} — ratio calculé par somme brute (approximation)
'..\data_constraints\MEIE_2050_HD.dat' généré pour l'année 2050.
[avertissement] unités mixtes pour industry/Chemicals : {'kt', 'GO (M$ 2016)'} — ratio calculé par somme brute (approximation)
'..\data_constraints\MEIE_2050_no_CCS.dat' généré pour l'année 2050.


In [16]:
# Aperçu du .dat généré pour 2050_HD
print(Path(OUTPUT_DIR / "MEIE_2050_HD.dat").read_text())

# CO2 emissions and capture
let co2_limit := 0.00 ;
let ccs_capture_limit := 25.00 ;
let ccs_sequestration_limit := 25.00 ;

# AGRICULTURE
let end_uses_demand_year['YEAR_2050','ELECTRICITY_MV','AGRICULTURE'] := 1976.53 ;
let end_uses_demand_year['YEAR_2050','HEAT_LOW_T_HW','AGRICULTURE'] := 124.40 ;
let end_uses_demand_year['YEAR_2050','HEAT_LOW_T_SH','AGRICULTURE'] := 1119.60 ;
let end_uses_demand_year['YEAR_2050','LIGHTING','AGRICULTURE'] := 82.36 ;
let end_uses_demand_year['YEAR_2050','MOBILITY_FREIGHT_SD','AGRICULTURE'] := 3147.02 ;
# HOUSEHOLDS
let end_uses_demand_year['YEAR_2050','ELECTRICITY_LV','HOUSEHOLDS'] := 23007.05 ;
let end_uses_demand_year['YEAR_2050','HEAT_LOW_T_HW','HOUSEHOLDS'] := 19278.15 ;
let end_uses_demand_year['YEAR_2050','HEAT_LOW_T_SC','HOUSEHOLDS'] := 3815.33 ;
let end_uses_demand_year['YEAR_2050','HEAT_LOW_T_SH','HOUSEHOLDS'] := 81422.02 ;
let end_uses_demand_year['YEAR_2050','LIGHTING','HOUSEHOLDS'] := 7053.55 ;
# INDUSTRY
let end_uses_demand_year['YEAR_205

## 5. Écriture du fichier `.mod` (capacités installées minimales)

En plus des demandes (`.dat`), certains scénarios imposent une capacité installée
minimale (`F_Mult`) pour certaines technologies de production. La variante à
utiliser vient de la colonne `capacite` de l'onglet `scenario` (table d'assemblage) ;
cette variante est ensuite recherchée dans l'onglet `capacites`. Le mapping colonne
Excel -> technologie AMPL -> nom de contrainte est défini ici, dans le notebook (pas
dans l'Excel).

In [17]:
CAPACITY_MAPPING = {
    # colonne Excel : (nom_technologie_AMPL, nom_contrainte_AMPL)
    "Hydro": ("NEW_HYDRO_DAM", "NEW_HYDRO_DAM"),
    "Éolien": ("WIND_ONSHORE", "NEW_WIND"),
    "PV Roof": ("PV_ROOF", "NEW_SOLAR_ROOF"),
    "PV Ground": ("PV_GROUND", "NEW_SOLAR_GROUND"),
    "Nucléaire": ("NUCLEAR", "NEW_NUCLEAR"),
}


def load_installed_capacity(scenario_name, scenario_path=SCENARIO_PATH):
    """Lit l'onglet `capacites` pour la variante `capacite` choisie pour ce scénario
    (via l'onglet `scenario`) et retourne les contraintes de capacité minimale
    (F_Mult) à écrire.

    Retourne un dict {nom_contrainte: (nom_technologie_AMPL, valeur)}. Les colonnes
    vides (NaN) pour cette variante sont ignorées : aucune contrainte n'est écrite.
    """
    assembly_row = get_scenario_assembly(scenario_name, scenario_path)
    variante = assembly_row.get("capacite")

    if pd.isna(variante):
        print(f"[avertissement] capacite : aucune variante choisie pour '{scenario_name}' — ignoré")
        return {}

    df = _read_scenario_sheet(scenario_path, "capacites")
    if variante not in df.index:
        print(f"[avertissement] capacites : pas de ligne pour la variante '{variante}' — ignoré")
        return {}

    row = df.loc[variante]
    constraints = {}
    for col_excel, (tech_ampl, constraint_name) in CAPACITY_MAPPING.items():
        if col_excel not in df.columns:
            continue
        valeur = row[col_excel]
        if pd.isna(valeur):
            continue
        constraints[constraint_name] = (tech_ampl, valeur)
    return constraints


In [18]:
def generer_mod_scenario(nom_fichier, annee, constraints, ccs_capture_limit=None):
    """
    nom_fichier         : chemin de sortie (.mod)
    annee               : année cible (ex. 2050) -> tag 'YEAR_2050'
    constraints         : dict {nom_contrainte: (nom_technologie_AMPL, valeur)}, comme
                           retourné par `load_installed_capacity`.
    ccs_capture_limit   : valeur scalaire [Mt CO2] (colonne 'Capture limite [Mt]' de
                           l'onglet contrainte_env, via `load_scenario_extra_params`) ou
                           None si aucune variante/valeur n'est définie pour ce scénario
                           -> dans ce cas, aucune contrainte de captage n'est écrite.

    Écrit un bloc `subject to ... : F_Mult[...] >= valeur ;` par contrainte de capacité,
    ex. :

        subject to NEW_HYDRO_DAM:
            F_Mult['YEAR_2050',"NEW_HYDRO_DAM"] >= 4.00;

    puis, si `ccs_capture_limit` est fourni, une contrainte de limite de captage de CO2 :

        param ccs_limit_capture default 15.00 >= 0;

        subject to co2_capture_limit_1:
            sum{t in PERIODS, i in RESOURCES union TECHNOLOGIES diff STORAGE_TECH:
                layers_in_out['YEAR_2050',i,"CO2_C"] > 0}
                (abs(layers_in_out['YEAR_2050',i,"CO2_C"]) * F_Mult_t['YEAR_2050',i,t] * t_op[t])
            <= ccs_limit_capture;
    """
    year_tag = f"YEAR_{annee}"

    with open(nom_fichier, "w", encoding="utf-8") as f:
        f.write("# Capacite installee minimale (F_Mult)\n\n")
        for constraint_name, (tech_ampl, valeur) in constraints.items():
            f.write(f"subject to {constraint_name}:\n")
            f.write(f"    F_Mult['{year_tag}',\"{tech_ampl}\"] >= {valeur:.2f};\n\n")

        if ccs_capture_limit is not None:
            f.write("# Limite de captage de CO2 (CCS)\n")
            f.write(f"param ccs_limit_capture default {ccs_capture_limit:.2f} >= 0;\n\n")
            f.write("subject to co2_capture_limit_1:\n")
            f.write("    sum{t in PERIODS, i in RESOURCES union TECHNOLOGIES diff STORAGE_TECH:\n")
            f.write(f"        layers_in_out['{year_tag}',i,\"CO2_C\"] > 0}}\n")
            f.write(f"        (abs(layers_in_out['{year_tag}',i,\"CO2_C\"]) * F_Mult_t['{year_tag}',i,t] * t_op[t])\n")
            f.write("    <= ccs_limit_capture;\n\n")

    n_ccs = 1 if ccs_capture_limit is not None else 0
    print(f"'{nom_fichier}' genere pour l'annee {annee} "
          f"({len(constraints)} contrainte(s) de capacite, {n_ccs} contrainte de captage CO2).")


def generate_scenario_mod(scenario_name, year, output_dir=OUTPUT_DIR, scenario_path=SCENARIO_PATH):
    """scenario_name : clé dans l'onglet `capacites` / `contrainte_env` (ex. 'MEIE_2050_HD') ;
    year : année AMPL cible (ex. 2050) -> tag 'YEAR_2050'. Équivalent de `generate_scenario_dat`
    pour les contraintes de capacité installée et de captage de CO2 (CCS)."""
    constraints = load_installed_capacity(scenario_name, scenario_path)

    extra = load_scenario_extra_params(scenario_name, scenario_path)
    ccs_capture_limit = extra.get("ccs_capture_limit")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{scenario_name}.mod"
    generer_mod_scenario(out_path, year, constraints, ccs_capture_limit)
    return out_path

### Génération pour tous les scénarios de l'onglet `scenario`

Même logique que pour les `.dat` : un fichier `.mod` par scénario, dans le même
dossier de sortie.

In [19]:
for scenario_name, row in scenarios.iterrows():
    generate_scenario_mod(scenario_name=scenario_name, year=int(row["year"]))


[avertissement] contrainte_env : aucune variante choisie pour '2023' — ignoré
'..\data_constraints\2023.mod' genere pour l'annee 2023 (0 contrainte(s) de capacite, 0 contrainte de captage CO2).
'..\data_constraints\MEIE_2050.mod' genere pour l'annee 2050 (0 contrainte(s) de capacite, 1 contrainte de captage CO2).
'..\data_constraints\MEIE_2050_HD.mod' genere pour l'annee 2050 (0 contrainte(s) de capacite, 1 contrainte de captage CO2).
'..\data_constraints\MEIE_2050_no_CCS.mod' genere pour l'annee 2050 (1 contrainte(s) de capacite, 1 contrainte de captage CO2).
